In [24]:
import nltk
nltk.download("punkt_tab")

from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import random

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pjhoor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [25]:
import pandas as pd

df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: object


In [26]:
# Dataset statistics before tokenization
emotions_unique = df["emotion"].unique()
print("Dataset statistics before tokenization for each unique emotion:")
print()
for emotion in emotions_unique:
    subset = df[df["emotion"] == emotion]
    N = len(subset["emotion"])
    print(f"{emotion} has {N} occurences, {round(N/len(df["emotion"]) * 100, 2)}%")
    average_length = subset["content"].str.len().mean()
    print(f"{emotion} has an average length of {average_length:.2f} characters")
    print()


Dataset statistics before tokenization for each unique emotion:

joy has 1092 occurences, 14.53%
joy has an average length of 103.14 characters

fear has 1076 occurences, 14.32%
fear has an average length of 123.96 characters

anger has 1079 occurences, 14.36%
anger has an average length of 130.65 characters

sadness has 1082 occurences, 14.4%
sadness has an average length of 103.93 characters

disgust has 1066 occurences, 14.18%
disgust has an average length of 114.61 characters

shame has 1071 occurences, 14.25%
shame has an average length of 116.73 characters

guilt has 1050 occurences, 13.97%
guilt has an average length of 124.89 characters



In [27]:
# Representation 1: Tokenization
df["tokens"] = df["content"].fillna("").apply(word_tokenize)

# Calculate and print the average number of tokens per sentence
average_token_length = df["tokens"].str.len().mean()

print(f"Average token length of a sentence: {average_token_length:.2f} tokens")

Average token length of a sentence: 24.57 tokens


In [28]:
# Representation 2: Bag of words
vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)

bag_matrix = vectorizer.fit_transform(df["tokens"].tolist())

bag_df = pd.DataFrame(bag_matrix.toarray(),
                      columns=vectorizer.get_feature_names_out(),
                      index=df.index
)

print(bag_df.head(5))

   !  $  %  &  '  ''  '84  'cos  'd  'll  ...  yr.  yrs  yukky  zero  \
0  0  0  0  0  0   0    0     0   0    0  ...    0    0      0     0   
1  0  0  0  0  0   0    0     0   0    0  ...    0    0      0     0   
2  0  0  0  0  0   0    0     0   0    0  ...    0    0      0     0   
3  0  0  0  0  0   0    0     0   0    0  ...    0    0      0     0   
4  0  0  0  0  0   0    0     0   0    0  ...    0    0      0     0   

   zig-zagging  zip  zipper  zombies  zone  zoophiliac  
0            0    0       0        0     0           0  
1            0    0       0        0     0           0  
2            0    0       0        0     0           0  
3            0    0       0        0     0           0  
4            0    0       0        0     0           0  

[5 rows x 10175 columns]


In [29]:
# Representation 3: Bigrams

bigrams = []
for sentence in df["tokens"]:
    bigrams_per_sentence = []
    for i in range(len(sentence) - 1):
        bigram = sentence[i] + " " + sentence[i+1]
        bigrams_per_sentence.append(bigram)
    bigrams.append(bigrams_per_sentence)
df["bigrams"] = bigrams

In [30]:
# Representation 4: Trigrams

trigrams = []
for sentence in df["tokens"]:
    trigrams_per_sentence = []
    for i in range(len(sentence) - 2):
        trigram = sentence[i] + " " + sentence[i+1] + " " + sentence[i+2]
        trigrams_per_sentence.append(trigram)
    trigrams.append(trigrams_per_sentence)
df["trigrams"] = trigrams

In [31]:
print(df["trigrams"][0])

['On days when', 'days when I', 'when I feel', 'I feel close', 'feel close to', 'close to my', 'to my partner', 'my partner and', 'partner and other', 'and other friends', 'other friends .', 'friends . When', '. When I', 'When I feel', 'I feel at', 'feel at peace', 'at peace with', 'peace with myself', 'with myself and', 'myself and also', 'and also experience', 'also experience a', 'experience a close', 'a close contact', 'close contact with', 'contact with people', 'with people whom', 'people whom I', 'whom I regard', 'I regard greatly', 'regard greatly .']


In [32]:
# Architecture 1: Log-Linear
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
texts = df["content"].fillna("")
labels = sorted(df["emotion"].unique())
label_to_id = {label: i for i, label in enumerate(labels)}
y = df["emotion"].map(label_to_id)


token_indicator_model = Pipeline([
    ("features", CountVectorizer(
        tokenizer=word_tokenize,
        token_pattern=None,
        binary=True
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        C=1.0
    ))
])

# I chose for StratifiedKFold to keep the even emotion distrubition of the dataset in every fold
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

results = cross_validate(
    token_indicator_model,
    texts,
    y,
    cv=cv,
    scoring=["accuracy"],
    return_train_score=True
)

print("Validation accuracy:", results["test_accuracy"])
print("Mean accuracy:", results["test_accuracy"].mean())

Validation accuracy: [0.58976064 0.56420492 0.57884232 0.56886228 0.58150366]
Mean accuracy: 0.5766347623901134


In [ ]:
# Cell to compare the three representations when used in logistic regression
models = {
    "token-indicator": Pipeline([
        ("features", CountVectorizer(
            tokenizer=word_tokenize,
            token_pattern=None,
            binary=True,
            ngram_range=(1, 1)
        )),
        ("classifier", LogisticRegression(max_iter=2000))
    ]),
    "bag-of-words": Pipeline([
        ("features", CountVectorizer(
            tokenizer=word_tokenize,
            token_pattern=None,
            binary=False,
            ngram_range=(1, 1)
        )),
        ("classifier", LogisticRegression(max_iter=2000))
    ]),
    "bigrams-and-trigrams": Pipeline([
        ("features", CountVectorizer(
            tokenizer=word_tokenize,
            token_pattern=None,
            binary=False,
            ngram_range=(2, 3)
        )),
        ("classifier", LogisticRegression(max_iter=2000))
    ]),
    "unigrams-bigrams-and-trigrams": Pipeline([ # I'm testing if the scores improve when adding unigrams
            ("features", CountVectorizer(
                tokenizer=word_tokenize,
                token_pattern=None,
                binary=False,
                ngram_range=(1, 3)
            )),
            ("classifier", LogisticRegression(max_iter=2000))
        ])
}

scores = {}

for name, model in models.items():
    result = cross_validate(
        model,
        texts,
        y,
        cv=cv,
        scoring=["accuracy"],
        return_train_score=True
    )
    scores[name] = result["test_accuracy"]
    print(
        f"{name}: "
        f"accuracy = {result['test_accuracy'].mean():.4f}, "
    )

token-indicator: accuracy = 0.5766, 
bag-of-words: accuracy = 0.5757, 
bigrams-and-trigrams: accuracy = 0.5249, 
unigrams-bigrams-and-trigrams: accuracy = 0.5805, 


In [35]:
# classes and functions for mlp
corpus = " ".join(df["content"].fillna("").astype(str))

vocab = sorted(set(word_tokenize(corpus)))
token_to_id = {token: i for i, token in enumerate(vocab)}

class Encoder:
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")


def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

class TokenIndicator(Encoder):
    def __init__(self, token_to_id):
        self.token_to_id = token_to_id

    def __len__(self):
        return len(self.token_to_id)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self), dtype=np.float32)

        for token in set(word_tokenize(x)):
            token_id = self.token_to_id.get(token)
            if token_id is not None:
                vector[token_id] = 1
        return vector
    
token_indicator_encoder = TokenIndicator(token_to_id)
doc_repr = token_indicator_encoder(corpus)

In [36]:
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [37]:
# Architecture 2: Single hidden layer MLP
import smlp

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)
accuracies = []
for fold in cv:
    metrics, best_epoch, best_pred = smlp.train_neural_network_model(
    cv[fold]["training"],
    cv[fold]["validation"],
    num_classes=len(labels),
    encoder=token_indicator_encoder,
    save_to=f"{results_folder}/mlp_{fold}_test",
    num_epochs=10)
    accuracies.append(best_pred)
print(accuracies)

NameError: name 'itertools' is not defined